In [ ]:
import pandas as pd
from pycaret.regression import *
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [ ]:
historic_data = pd.read_csv('data/combined_data/historical/historical_lakevic_data.csv')
#s = setup(data=historic_data, target='runoff', session_id=123)

,Description,Value
0,Session id,123
1,Target,runoff
2,Target type,Regression
3,Original data shape,"(1216, 5)"
4,Transformed data shape,"(1216, 19)"
5,Transformed train set shape,"(851, 19)"
6,Transformed test set shape,"(365, 19)"
7,Numeric features,2
8,Categorical features,2
9,Rows with missing values,25.0%


In [ ]:
numerical_features = ['year', 'rainfall']
categorical_features = ['month', 'region']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    #('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numerical_features),
        ('cat_transform', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
        ],
    remainder='passthrough'
)

In [19]:
best = compare_models(sort='MAE', exclude=['lightgbm'])
print(best)

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
gbr,Gradient Boosting Regressor,227.1357,246853.0582,485.7421,0.3013,2.1471,937.8087,0.0220
et,Extra Trees Regressor,227.5635,272593.6821,499.7068,0.1447,1.7807,263.8666,0.0360
rf,Random Forest Regressor,230.2364,271552.2150,505.3046,0.1719,1.7965,300.3907,0.0470
dt,Decision Tree Regressor,257.5961,357518.3237,584.5546,-0.1430,1.8935,70.2671,0.0130
knn,K Neighbors Regressor,267.7487,322509.9891,554.9902,0.1191,2.0674,531.8873,0.0140
huber,Huber Regressor,272.7501,422966.2741,616.1167,0.1293,2.2098,743.6153,0.0150
en,Elastic Net,333.8946,345601.4152,569.0836,0.1852,2.7572,2990.1065,0.0110
br,Bayesian Ridge,341.8421,331285.6699,557.7096,0.1983,2.8468,3279.5104,0.0110
omp,Orthogonal Matching Pursuit,344.0757,356215.6617,579.2019,0.1489,2.7822,3033.9582,0.0110
lasso,Lasso Regression,345.4650,331307.4501,557.8867,0.1924,2.8617,3330.7374,0.0560


GradientBoostingRegressor(random_state=123)


In [ ]:
evaluate_model(best)

interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [60]:
"""
    I'm just trying to see how well the predictions from the gbr model align with the projected runoffs.
    I'm not sure though if this is the best way to do it.
"""

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

#projected_data is the data that has been projected with runoffs
projected_data = pd.read_csv('data/combined_data/projected/projected_lakevic_data.csv')
# print(projected_data.describe())

#predictionData is the data that will be used to predict the runoffs and therefore does not include the runoff column.
predictionData = projected_data.drop(columns=['runoff'])
gbrPrediction = predict_model(best, data=predictionData)
gbrPrediction.rename(columns={'prediction_label': 'runoff'}, inplace=True)
comp_df = pd.merge(projected_data, gbrPrediction, on=['year', 'month', 'region'], suffixes=('_actual', '_pred'))
#print(gbrPrediction)
print(comp_df)
print("\nThe Mean Absolute Error : %d " %(mean_absolute_error(comp_df['runoff_actual'], comp_df['runoff_pred'])))  # MAE
print("The Mean Squared Error : %d " %(mean_squared_error(comp_df['runoff_actual'], comp_df['runoff_pred'])))  # MSE
print("The R2 Score : %f " %(r2_score(comp_df['runoff_actual'], comp_df['runoff_pred'])))  # R2


     year  month         region  rainfall_actual  runoff_actual  \
0    2026  April         Kagera            5.478       1381.322   
1    2026  April  Lake Victoria           11.460        708.806   
2    2026  April         Simiyu            4.534        708.806   
3    2026  April  Victoria Nile              NaN       3611.252   
4    2026   Aug.         Kagera            0.298         85.005   
..    ...    ...            ...              ...            ...   
475  2035   Oct.  Victoria Nile              NaN       3204.099   
476  2035  Sept.         Kagera            3.839        250.111   
477  2035  Sept.  Lake Victoria            3.825        183.906   
478  2035  Sept.         Simiyu            9.079        183.906   
479  2035  Sept.  Victoria Nile              NaN       1474.566   

     rainfall_pred  runoff_pred  
0            5.478   798.347180  
1           11.460  1168.612164  
2            4.534   831.522967  
3              NaN  1837.428064  
4            0.298    63.